# Funciones en Python aplicadas a Oil & Gas

## Estimación de producción mediante la ecuación de Vogel

En ingeniería de producción, la relación de comportamiento de afluencia (*Inflow Performance Relationship, IPR*) permite estimar el caudal que puede entregar un pozo para diferentes valores de presión de fondo fluyente.

La ecuación de Vogel es un modelo empírico utilizado para representar el comportamiento de pozos de petróleo bajo condiciones de flujo bifásico. En este ejercicio se mantendrán constantes la presión del yacimiento y el caudal máximo, mientras se modifica la presión de fondo fluyente.

$$
q_o = q_{max}
\left[
1-0.2\left(\frac{P_{wf}}{P_r}\right)
-0.8\left(\frac{P_{wf}}{P_r}\right)^2
\right]
$$

Donde:

- $q_o$: caudal estimado de petróleo, BPD.
- $q_{max}$: caudal máximo teórico del pozo, BPD.
- $P_r$: presión promedio del yacimiento, psi.
- $P_{wf}$: presión de fondo fluyente, psi.

Cuando $P_{wf}=P_r$, el caudal estimado tiende a cero. Al disminuir $P_{wf}$, aumenta el potencial de producción. En el límite $P_{wf}=0$, el modelo alcanza $q_{max}$.


## 1. Construcción de la función

La función encapsula el cálculo para reutilizarlo con diferentes condiciones de presión.

Se utilizarán dos parámetros obligatorios, dos parámetros con valor por defecto y `return` para entregar el resultado calculado.


In [ ]:
def calcular_caudal_vogel(caudal_maximo, presion_yacimiento, presion_fondo=0, decimales=2):
    """
    Calcula el caudal de petróleo mediante la ecuación de Vogel.

    Parámetros:
    caudal_maximo (float): Caudal máximo teórico del pozo, BPD.
    presion_yacimiento (float): Presión promedio del yacimiento, psi.
    presion_fondo (float): Presión de fondo fluyente, psi.
    decimales (int): Número de decimales del resultado.

    Retorna:
    float: Caudal estimado de petróleo, BPD.
    """

    relacion_presion = presion_fondo / presion_yacimiento

    caudal = caudal_maximo * (
        1 - 0.2 * relacion_presion - 0.8 * relacion_presion**2
    )

    return round(caudal, decimales)


## 2. Evaluación de una condición puntual

Condiciones del ejemplo:

- $q_{max}=1200$ BPD
- $P_r=3000$ psi
- $P_{wf}=1500$ psi


In [ ]:
caudal = calcular_caudal_vogel(1200, 3000, 1500)

print(f"Caudal estimado: {caudal} BPD")


### Argumentos por posición

Cuando los valores se envían por posición, Python los asigna según el orden definido en la función:

```python
calcular_caudal_vogel(1200, 3000, 1500)
```

corresponde a:

```text
1200  -> caudal_maximo
3000  -> presion_yacimiento
1500  -> presion_fondo
```


### Argumentos por nombre

También se pueden declarar explícitamente los parámetros. Esto mejora la legibilidad y permite cambiar su orden al llamar la función.


In [ ]:
caudal = calcular_caudal_vogel(
    presion_fondo=1500,
    caudal_maximo=1200,
    presion_yacimiento=3000
)

print(f"Caudal estimado: {caudal} BPD")


## 3. Variación de la presión de fondo

Una aplicación práctica consiste en mantener constantes $P_r$ y $q_{max}$ y evaluar diferentes valores de $P_{wf}$.

Esto permite observar cómo cambia el potencial de producción del pozo a medida que cambia la presión de fondo fluyente.


In [ ]:
caudal_maximo = 1200
presion_yacimiento = 3000

presiones_fondo = [3000, 2500, 2000, 1500, 1000, 500, 0]

for presion in presiones_fondo:
    caudal = calcular_caudal_vogel(
        caudal_maximo,
        presion_yacimiento,
        presion
    )

    print(f"Pwf = {presion:4} psi  ->  q = {caudal:7.2f} BPD")


## 4. Construcción de la curva IPR

Para representar la curva completa se generará un conjunto de valores de presión entre 0 y la presión del yacimiento.

Cada valor de presión se enviará a la misma función y los resultados se almacenarán en una tabla para su visualización.


In [ ]:
import numpy as np
import pandas as pd

caudal_maximo = 1200
presion_yacimiento = 3000

presiones_fondo = np.linspace(0, presion_yacimiento, 50)

caudales = [
    calcular_caudal_vogel(
        caudal_maximo,
        presion_yacimiento,
        presion
    )
    for presion in presiones_fondo
]

df_ipr = pd.DataFrame({
    "Presion_fondo_psi": presiones_fondo,
    "Caudal_BPD": caudales
})

df_ipr.head()


## 5. Visualización con Plotly

La curva IPR muestra la relación entre la presión de fondo fluyente y el caudal estimado. Los marcadores permiten visualizar las condiciones calculadas sobre la curva.


In [ ]:
import plotly.express as px

fig = px.line(
    df_ipr,
    x="Caudal_BPD",
    y="Presion_fondo_psi",
    markers=True,
    labels={
        "Caudal_BPD": "Caudal de petróleo (BPD)",
        "Presion_fondo_psi": "Presión de fondo fluyente, Pwf (psi)"
    },
    title="Curva IPR de Vogel"
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig.show()


## 6. Interpretación

La curva permite analizar diferentes condiciones de operación del pozo:

- con $P_{wf}$ cercana a $P_r$, el diferencial de presión es pequeño y el caudal tiende a cero;
- al disminuir $P_{wf}$, aumenta el caudal estimado;
- cuando $P_{wf}=0$, el modelo alcanza el caudal máximo teórico $q_{max}$.

El objetivo del ejercicio es demostrar cómo una función permite reutilizar una misma lógica de ingeniería para evaluar múltiples escenarios sin repetir manualmente la ecuación.


## 7. Ejercicio propuesto

Modifique los valores de `presion_fondo`, `presion_yacimiento` y `caudal_maximo`.

Luego genere nuevamente la curva IPR y compare cómo cambian las condiciones de producción del pozo.
